# Feedback Storage Comparison


## 1) Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2) Install Dependencies


In [ ]:
!pip install -q pandas numpy matplotlib seaborn


## 3) Load Dataset


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = '/content/drive/MyDrive/Deep Learning Project/AI Agentic/data/processed'
required = ['X_train.npy','X_test.npy','y_train.npy','y_test.npy']
print('Data path:', DATA_PATH)
print('Files:', os.listdir(DATA_PATH))
missing = [f for f in required if not os.path.exists(os.path.join(DATA_PATH,f))]
if missing:
    raise FileNotFoundError(f'Missing required files: {missing}')

import shutil, json, sqlite3, time
from datetime import datetime, timezone
X_test=np.load(os.path.join(DATA_PATH,'X_test.npy'))
y_test=np.load(os.path.join(DATA_PATH,'y_test.npy')).reshape(-1)
if X_test.ndim==3 and X_test.shape[-1]==1: X_test=X_test[...,0]



## 4) Simulate Attack Detection Results


In [ ]:
def sim(X,y,n=500,seed=42):
    rng=np.random.default_rng(seed)
    idx=rng.choice(len(y),size=min(n,len(y)),replace=False)
    amap={0:'BENIGN',1:'DOS',2:'PORTSCAN',3:'BRUTEFORCE',4:'DDOS'}
    rows=[]
    for sid,i in enumerate(idx):
        yi=int(y[i]); atk=amap.get(yi,f'ATTACK_{yi}')
        risk=float(np.clip(0.6*np.mean(np.abs(X[i]))+0.3*(yi>0)+rng.normal(0,0.08),0,1))
        dec='BLOCK_IP' if risk>0.8 else 'RATE_LIMIT' if risk>0.6 else 'ALERT_ADMIN' if risk>0.4 else 'ALLOW'
        hf='confirmed_attack' if atk!='BENIGN' else ('false_alarm' if dec!='ALLOW' else 'confirmed_benign')
        rows.append({'timestamp':datetime.now(timezone.utc).isoformat(),'sample_id':sid,'attack_type':atk,'risk_score':risk,'decision':dec,'human_feedback':hf,'features':X[i][:32].astype(float).tolist()})
    return rows
records=sim(X_test,y_test)
len(records)



## 5) Storage Methods + Benchmark


In [ ]:
class JS:
    def __init__(self,p): self.p=p; os.makedirs(os.path.dirname(p),exist_ok=True)
    def store(self,r):
        with open(self.p,'a',encoding='utf-8') as f:f.write(json.dumps(r,ensure_ascii=True)+'\n')
    def get(self,n=100):
        with open(self.p,'r',encoding='utf-8') as f: return [json.loads(x) for x in f.readlines()[-n:]]
class SQ:
    def __init__(self,p):
        self.c=sqlite3.connect(p)
        self.c.execute('CREATE TABLE IF NOT EXISTS feedback(ts TEXT,sid INT,atk TEXT,risk REAL,decision TEXT,hf TEXT)'); self.c.commit()
    def store(self,r):
        self.c.execute('INSERT INTO feedback VALUES (?,?,?,?,?,?)',(r['timestamp'],r['sample_id'],r['attack_type'],r['risk_score'],r['decision'],r['human_feedback'])); self.c.commit()
    def get(self,n=100):
        q=self.c.execute('SELECT ts,sid,atk,risk,decision,hf FROM feedback ORDER BY rowid DESC LIMIT ?', (int(n),)).fetchall()
        return q
class VM:
    def __init__(self,p): self.vp=p+'_v.npy'; self.mp=p+'_m.jsonl'; self.vec=[]; self.meta=[]; os.makedirs(os.path.dirname(self.vp),exist_ok=True)
    def emb(self,ft):
        a=np.asarray(ft,dtype=np.float32)
        return np.array([a.mean(),a.std(),a.max(),a.min(),np.median(a),np.percentile(a,75),np.percentile(a,25),np.linalg.norm(a)],dtype=np.float32)
    def store(self,r): self.vec.append(self.emb(r['features'])); self.meta.append({k:r[k] for k in ['timestamp','sample_id','attack_type','risk_score','decision','human_feedback']})
    def finalize(self):
        np.save(self.vp,np.vstack(self.vec))
        with open(self.mp,'w',encoding='utf-8') as f:
            for m in self.meta: f.write(json.dumps(m,ensure_ascii=True)+'\n')
    def get(self,n=100): return self.meta[-n:]

wd='/content/feedback_storage_benchmark'
if os.path.exists(wd): shutil.rmtree(wd)
os.makedirs(wd,exist_ok=True)
rows=[]

js=JS(os.path.join(wd,'feedback.jsonl')); t=time.perf_counter(); [js.store(r) for r in records]; st=time.perf_counter()-t
t=time.perf_counter(); _=js.get(200); rt=time.perf_counter()-t
t=time.perf_counter(); [js.get(50) for _ in range(200)]; ql=(time.perf_counter()-t)/200
rows.append({'Method':'JSON Log Storage','StorageTimeSec':st,'RetrievalTimeSec':rt,'QueryLatencySec':ql})

sq=SQ(os.path.join(wd,'feedback.db')); t=time.perf_counter(); [sq.store(r) for r in records]; st=time.perf_counter()-t
t=time.perf_counter(); _=sq.get(200); rt=time.perf_counter()-t
t=time.perf_counter(); [sq.get(50) for _ in range(200)]; ql=(time.perf_counter()-t)/200
rows.append({'Method':'SQLite Database Storage','StorageTimeSec':st,'RetrievalTimeSec':rt,'QueryLatencySec':ql})

vm=VM(os.path.join(wd,'feedback_vec')); t=time.perf_counter(); [vm.store(r) for r in records]; vm.finalize(); st=time.perf_counter()-t
t=time.perf_counter(); _=vm.get(200); rt=time.perf_counter()-t
t=time.perf_counter(); [vm.get(50) for _ in range(200)]; ql=(time.perf_counter()-t)/200
rows.append({'Method':'Vector Memory Storage','StorageTimeSec':st,'RetrievalTimeSec':rt,'QueryLatencySec':ql})

res=pd.DataFrame(rows).sort_values('QueryLatencySec').reset_index(drop=True)
res



## 6) Charts + Best


In [ ]:
res.plot(x='Method',y='StorageTimeSec',kind='bar',figsize=(9,4),title='Storage Time')
plt.tight_layout(); plt.show()
res.plot(x='Method',y='RetrievalTimeSec',kind='bar',figsize=(9,4),title='Retrieval Time')
plt.tight_layout(); plt.show()
res.plot(x='Method',y='QueryLatencySec',kind='bar',figsize=(9,4),title='Query Latency')
plt.tight_layout(); plt.show()
print('Best method by query latency:',res.iloc[0]['Method'])

